# Function 2 - Week 1: initial data exploration

Ten supplied observations, two inputs (`x1` and `x2`), and larger outputs are better. Both inputs lie in `[0, 1)`.

The [course description](https://classroom.emeritus.org/courses/18642/modules/items/3237303) characterises this function as a **noisy log-likelihood with local optima**. That is useful context; these ten observations do not tell us the underlying formula or the size of the noise.

**Taper-waper: start by reducing ignorance, finish by exploiting what we have learned.**

In [ ]:
from pathlib import Path
import sys

stage = next(p / "stage-2-bbo" for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "stage-2-bbo/src").is_dir())
sys.path.insert(0, str(stage / "src"))
from initial_exploration import load_initial_data, preview, plot_ranked_locations, plot_sampling_gaps, plot_ordered_outputs

data = load_initial_data(stage, function_id=2)
figures = stage.parent / ".runtime/function-2-initial-analysis"
from function_2_selection import (analyse_function_2_selection, plot_hypothesis_surfaces, plot_selection_heatmaps, selection_balance_table, selection_noise_table, selection_candidate_table, selection_fit_table, selection_findings, selection_sensitivity_findings)

## A quick look at the values

All ten observations, in their supplied order, rounded to six decimal places for display. **Rank 1 is the highest observed output; rank 10 is the lowest.** The ranks are not row IDs.

In [ ]:
preview(data, rows=len(data), output_format="{:.6f}")

The outputs range from **−0.065624 to 0.611205**. Eight are positive and two are negative; none is exactly zero. Their scale is already easy to read.

## Where the observations sit

Marker numbers and colours show output rank. They tell us the ordering of the observations, not how far apart their output values are.

In [ ]:
plot_ranked_locations(data, save_to=figures / "ranked-locations.png")

The highest observed output is **0.611205 at (0.702637, 0.926564)**. The runner-up is **0.538996 at (0.665800, 0.123969)**, about 0.803 input-space units away. High observed values occur near both the top and bottom of the square, rather than forming one tight cluster.

The four highest outputs all have `x1` above 0.66. That is a pattern in this sample, but `x2` also changes between observations, so it does not isolate the effect of `x1`.

The closest pair is ranks **3 and 4**, about **0.075** units apart, with outputs **0.420586** and **0.293993**. Their difference could reflect local variation, noise, or both.

## Sampling gaps

Brighter areas are farther from the nearest observation. This heat map describes **sampling distance**, not predicted output or model uncertainty. The marker numbers retain their output ranks.

In [ ]:
plot_sampling_gaps(data, save_to=figures / "sampling-gaps.png")

The largest gap reaches into the **upper-left corner**. The left side of the square is sparsely sampled, and there is also room between the lower observations and those in the upper half.

The two highest outputs are already far apart. Together with the unsampled areas, that leaves room for several different explanations of the surface; the location plot alone cannot establish how many peaks there are.

## How large are the outputs?

Both panels order the observations from best to worst. The left keeps the supplied values. The right shows `log₁₀ |y|`, with colour and marker shape retaining the sign, for comparison with Function 1.

In [ ]:
plot_ordered_outputs(data, save_to=figures / "ordered-outputs.png")

The largest absolute output is about **44 times** the smallest: roughly **1.64 orders of magnitude**. The original scale shows the differences clearly. No single value compresses the remaining observations into an unreadable line, so all ten remain in these charts.

The logarithmic-magnitude panel emphasises small magnitudes, but its height is **not a ranking of quality**: a negative value becomes a positive magnitude before taking the logarithm. The course already describes the supplied output as a log-likelihood, so this second transformation is only a display comparison. There is no clear need to transform the outputs for this first exploration.

## Initial observations

Function 2 gives us visible differences between observations without needing a special scale. Its best two observations are widely separated, while the upper-left region remains largely unexplored.

Each input location appears only once. We therefore cannot estimate repeat-evaluation noise from these data or attach confidence to the observed ordering. **0.611205 is the best value supplied, not an established target or ceiling.**

## Two heatmaps, two questions

We can make the next-query decision explicit by combining **coverage benefit**
with **the ability to distinguish two explanations of the x1 pattern**.

**H1: an x1-only ridge.** The response is a function of x1 plus noise. A promising
value of x1 remains promising as x2 changes. This does not assume that increasing
x1 always improves the output.

**H2: local two-input features.** Good responses may be confined near particular
combinations of x1 and x2. Moving away in either input can change the response.

Both are fitted with Gaussian processes using a Matérn 3/2 kernel. H1 uses x1
alone. H2 initially uses both inputs with the same length scale, making local
features possible without prescribing a vertical ridge. We later relax that
restriction. These are working explanations, not an exhaustive set of possible
functions. [Gaussian-process method](https://scikit-learn.org/stable/modules/gaussian_process.html).

For the main comparison, the assumed observation-noise standard deviation is
**0.05 output units**. This is a sensitivity assumption, not a measured noise
level; we also examine 0.025, 0.10 and 0.15.

In [ ]:
selection = analyse_function_2_selection(data, output=figures)

In [ ]:
plot_hypothesis_surfaces(data, selection, save_to=figures / 'hypothesis-surfaces.png')

The panels share an output colour scale. White circles retain the original
output ranks. These are fitted mean surfaces; the local model's return towards
the sample mean far from observations is a modelling assumption, not evidence
of a flat true surface.

### Coverage benefit

Let S be the existing input locations, q a candidate, and d̄(S) the average
distance to the nearest observation over the unit square. Define

**C(q) = 1 − d̄(S ∪ {q}) / d̄(S).**

This is the fractional reduction in average sampling distance after adding q,
using the same geometric integral as Function 1. It rewards the area helped by
a query, rather than just its distance from the nearest existing point. Divide
by the best attainable C to put coverage on a 0–1 scale.

### Hypothesis discrimination

At q, let μ₁ and μ₂ be the two predicted means and v₁ and v₂ their predictive
variances **for a new noisy observation**. Define

**D(q) = (μ₁(q) − μ₂(q))² / (v₁(q) + v₂(q)).**

The score grows when predictions separate and shrinks when their uncertainty
or observation noise could obscure the difference. It is unchanged by a change
of output units. The sum of variances is a practical measure of combined spread;
the fits share training data, so this is not an independent-estimates test.

Map it to a bounded score with **H(q) = D(q) / (1 + D(q))**. D = 1 gives H = 0.5;
D = 4 gives H = 0.8. This fixed mapping preserves weak disagreement across model
and noise assumptions. Rescaling each map by its own maximum would make even
tiny disagreements look strong.

This is a heuristic for designing an informative query, inspired by
[selection through model disagreement](https://proceedings.mlr.press/v16/borisov11a/borisov11a.pdf).
H is not a p-value, a probability that a hypothesis is true, or a calculated
expected information gain.

### Mix the scores

**A<sub>w</sub>(q) = w · C(q)/C<sub>max</sub> + (1 − w) · H(q).**

Choose q to maximise A. Here w is the coverage weight: 1 gives pure coverage,
0 gives pure discrimination, and 0.5 is an explicit equal-weight example.
The weight expresses our priorities; it is not learned from the ten observations.

In [ ]:
plot_selection_heatmaps(data, selection, save_to=figures / 'coverage-and-hypotheses.png')

Each star marks the numerical maximum of that panel's score. The three maps
share a fixed 0–1 colour scale. The first two answer different learning questions;
the third shows their equal-weight blend.

In [ ]:
selection_balance_table(selection)

In [ ]:
selection_findings(selection)

In [ ]:
selection_candidate_table(selection)

## How much does the answer depend on our assumptions?

For each noise assumption, refit both explanations and optimise the 50:50 blend.
The final column retains the raw peak disagreement, so falling signal separation
does not disappear behind normalisation.

In [ ]:
selection_noise_table(selection)

For a small predictive check, leave out each observation in turn, refit on
the other nine, and predict the missing value. The table reports root mean squared
error (RMSE) in the original output units; lower is better. The mean baseline also
uses only the other nine values.

In [ ]:
selection_fit_table(selection)

In [ ]:
selection_sensitivity_findings(selection)

The numerical search evaluates an 81 × 81 candidate grid and refines its
local maxima, including boundary maxima. Coverage is integrated geometrically;
the candidate grid starts the search and draws the maps. Six-decimal neighbours
are checked when rounding a candidate. These are numerical recommendations
under stated assumptions, not a proof of global optimality.

<details><summary>Fit settings</summary>

Each fit standardises its training outputs internally, adjusts the assumed
noise variance to those units, and returns predictions in original units.
The signal variance is bounded to [0.05, 10] in standardised units and length
scales to [0.03, 2]. Four additional optimiser starts use seed 7. The two models
use the same fixed noise assumption in each comparison. Noise is added to the
latent prediction variance when computing D. Fitted-model uncertainty here is
conditional on estimated parameters, rather than integrating over them.

</details>

**Status: comparison complete; the blend weight and next query remain choices
to make. No portal query has been submitted.**

## The taper in Function 2

Across the 13 evaluation rounds, the emphasis shifts from exploration towards
fine tuning. Early queries reduce consequential blind spots and test competing
explanations. Middle rounds concentrate on promising regions while checking
alternatives. Later rounds refine the strongest candidates and assess whether
apparent gains are credible in the presence of noise.

The taper responds to evidence: an unexpected result can justify renewed
exploration, and each function can progress at a different pace. Coverage is one
way to reduce ignorance; it does not measure the probability of missing the
optimum. The recurring question is **what uncertainty does this query address,
and why is resolving it worth one of the remaining attempts?**

This function offers a different starting point from Function 1: its output
differences are readily visible, so fitting and coverage can both inform the
early decisions. Two uncertainties stand out. The upper-left gap could conceal
a good region. Meanwhile, the separated high observations near `x1 = 0.7` might
belong to a connected promising region, or to isolated features.

A query around **(0.7, 0.4)** could test that connection while filling part of the
middle gap. **(0.8, 0.4)** explores farther to the right. Comparing those regions
with the upper-left makes the trade-off concrete: predicted output, disagreement
between plausible fits, and geometric coverage each address a different concern.
These are candidate regions for comparison, not a settled submission.

The first query need not choose between learning and seeking a good output:
one observation can serve both purposes. As the evidence strengthens, more of
the remaining budget can go towards fine tuning. The course's noise description
makes it important to distinguish a credible gain from a favourable fluctuation.

The two heatmap terms both reward learning. Reducing the coverage weight only
shifts attention towards distinguishing models; it is not yet exploitation.
Later in the taper, selection will also need to reward credible high output
once the accumulated evidence supports that shift.

## What changes as dimensions increase?

This two-input case is our baseline: one two-dimensional plot shows every input
coordinate. With three inputs there are **3 coordinate pairs**; with eight there
are **28**. Those plots show projections. Points that look close in one projection
can be far apart in the full input space, so pairwise maps cannot establish full
coverage.

Even a toy grid with five positions per input grows from **25 combinations in
two dimensions** to **125 in three** and **390,625 in eight**. That is an
illustration of the growth, not a proposed sampling plan. As dimensions increase,
our comparison will track how the reason for a query changes: filling a gap,
testing an interaction between inputs, distinguishing fitted explanations, or
refining a promising region. Any fitted emphasis on particular inputs will need
evidence; higher dimension alone does not justify ignoring the others.

The later functions also have different behaviours and numbers of starting
observations. Differences in our approach cannot automatically be attributed to
dimension. The pattern to watch is **how the uncertainty worth spending a query
on changes**, while the taper remains our organising principle.